# 02 — Data Preprocessing

This notebook validates the target, checks data quality, and prepares a stratified train/test split. Encoding is fitted within the modeling pipeline to reduce data leakage.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path("../data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Read data/README.md and place the permitted CSV there."
    )
df = pd.read_csv(DATA_PATH)
if "Attrition" not in df.columns:
    raise ValueError("Expected target column 'Attrition'. Check data/README.md.")
print(f"Dataset shape: {df.shape}")
display(df.head())


Dataset shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [4]:
from sklearn.model_selection import train_test_split

# Remove exact duplicate rows, while recording how many were removed.
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f"Exact duplicate rows removed: {duplicate_count}")

# Normalize target labels.
target_map = {"yes": 1, "no": 0, "1": 1, "0": 0, "true": 1, "false": 0}
y = df["Attrition"].astype(str).str.strip().str.lower().map(target_map)
if y.isna().any():
    raise ValueError(f"Unexpected Attrition values: {df.loc[y.isna(), 'Attrition'].unique().tolist()}")

X = df.drop(columns=["Attrition"]).copy()

# Remove constant columns, if any; keep the list for reporting.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Removing constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

# Common benchmark dataset identifier, not a predictive feature.
for id_col in ["EmployeeNumber"]:
    if id_col in X.columns:
        X = X.drop(columns=id_col)

X_train, X_test, y_train, y_test = train_test_split(
    X, y.astype(int), test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train attrition rate:", round(float(y_train.mean()), 3))
print("Test attrition rate:", round(float(y_test.mean()), 3))


Exact duplicate rows removed: 0
Removing constant columns: ['EmployeeCount', 'Over18', 'StandardHours']
Train: (1176, 30) Test: (294, 30)
Train attrition rate: 0.162
Test attrition rate: 0.16


## Feature types

The modeling notebook builds preprocessing inside a scikit-learn Pipeline. That ensures category encoding and numeric scaling are learned from training data only.

In [5]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
Categorical features: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


## Save a local split summary (optional)

In [8]:
from pathlib import Path
Path("reports").mkdir(exist_ok=True)
summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(X_train), len(X_test)],
    "attrition_rate": [y_train.mean(), y_test.mean()]
})
display(summary.round(4))
summary.to_csv("../reports/data_split_summary.csv", index=False)


,split,rows,attrition_rate
0,train,1176,0.1616
1,test,294,0.1599
